# Colab Setup
Run this notebook top to bottom every session. It restores your work from Drive if a backup exists (so a runtime disconnect doesn't cost you the ~26 min preprocessing run again), otherwise downloads and processes fresh.

In [1]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Clone the repo
import os

REPO_DIR = "/content/Review-Summarization-LLM"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone https://github.com/jrsheffie/Review-Summarization-LLM.git {REPO_DIR}
    %cd {REPO_DIR}

/content
Cloning into 'Review-Summarization-LLM'...
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 106 (delta 41), reused 49 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 66.00 KiB | 3.14 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Review-Summarization-LLM


In [3]:
# 3. Install dependencies
!pip install -r requirements.txt --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 20.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.8 MB/s eta 0:00:00


In [ ]:
#4. Load API keys and git identity from Colab Secrets
import os
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
github_token = userdata.get('GITHUB_TOKEN')

!git config --global user.email "jrsheffie@gmail.com"
!git config --global user.name "jrsheffie"

print("API keys and git identity loaded.")

In [5]:
# 5. # Try restoring raw data + processed data from Drive backup first.
# If no backup exists yet, this just prints a message and does nothing --
# proceed to cells 6-7 to download and process from scratch.
import os

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
restored = False

if os.path.exists(f'{BACKUP_DIR}/clean_reviews.csv'):
    print('Found backup -- restoring processed data...')
    !mkdir -p data/processed data/raw
    !cp {BACKUP_DIR}/clean_reviews.csv {BACKUP_DIR}/train.csv {BACKUP_DIR}/val.csv {BACKUP_DIR}/test.csv {BACKUP_DIR}/product_batches.json data/processed/ 2>/dev/null
    !cp {BACKUP_DIR}/Reviews.csv data/raw/ 2>/dev/null
    restored = True
    print('Restored from Drive backup.')
else:
    print('No Drive backup found yet -- run the download + preprocessing cells below.')

Found backup -- restoring processed data...
Restored from Drive backup.


In [4]:
# 6. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — go to Runtime > Change runtime type > GPU (T4)")

CUDA available: True
Device: Tesla T4


## Only run cells 6-8 if cell 5 printed "No Drive backup found" -- otherwise skip to cell 9

In [6]:
from getpass import getpass
import os

token = getpass("Paste your Kaggle API token: ").strip()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token)

!python data/download_dataset.py

Paste your Kaggle API token: ··········
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.9MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [7]:
# 6. Set up Kaggle API and download dataset (skip if restored above)
# Upload your kaggle.json via the file browser on the left first, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:03<00:00, 34.2MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [8]:
# 7. Run preprocessing (skip if restored above -- this takes ~20-30 min on the full dataset)
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

2026-07-19 18:47:57,960 [INFO] Loaded 568454 rows from data/raw/Reviews.csv
2026-07-19 18:47:59,184 [INFO] drop_missing_and_nulls: 568454 -> 568454 rows
2026-07-19 18:48:00,616 [INFO] remove_duplicates: 568454 -> 567554 rows
2026-07-19 18:48:29,586 [INFO] filter_uninformative (min_words=5): 567554 -> 567534 rows
object address  : 0x7d90fbaae3e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [9]:
# 8. IMPORTANT: back this up to Drive immediately so you never redo steps 6-7 again
!mkdir -p {BACKUP_DIR}
!cp data/processed/*.csv data/processed/*.json {BACKUP_DIR}/
!cp data/raw/Reviews.csv {BACKUP_DIR}/
print('Backed up to Drive.')

^C
^C
Backed up to Drive.


## Everyone runs from here down

In [6]:
# 9. Exploratory data analysis on the real cleaned data
!python data/exploratory_analysis.py --input data/processed/clean_reviews.csv --output-dir outputs/figures

Total reviews: 536837
Unique products: 43824

Missing values per column:
Id                         0
ProductId                  0
UserId                     0
ProfileName               25
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

Average review length: 79.9 words

Figures saved to outputs/figures


In [7]:
# 10. BART baseline test -- pretrained model, no fine-tuning yet.
# Proves the model implementation works (Milestone 3 requirement).
from transformers import BartForConditionalGeneration, BartTokenizer
import pandas as pd

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')

df = pd.read_csv('data/processed/clean_reviews.csv')
sample_reviews = df['Text'].sample(3, random_state=1).tolist()

for i, review in enumerate(sample_reviews):
    inputs = tokenizer(review, return_tensors='pt', max_length=1024, truncation=True)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=60, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f'--- Review {i+1} ---')
    print('Original:', review[:200], '...')
    print('BART summary:', summary)
    print()

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

--- Review 1 ---
Original: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! ...
BART summary: Pedicure dog food is a decently good dog food. Our dogs love the meat and econally priced at twenty-four can package!!! Will buy again!!!!!! will buy again. Will buy another 24 can package of Pedicure Dog Food for our dogs.

--- Review 2 ---
Original: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it ...
BART summary: This won the Award for the Healthiest Oatmeal! I can see why 12g of Protein and it taste great! No added sugar or salt and it is so creamy it is by far the best instant Oatmeal I have ever had! And it hold me over past lunch. Love

--- Review 3 ---
Original: These bars are great for a snack. They have alot of healthy ingredients and low suga

In [9]:
# 11. Prompted LLM test -- uses Colab Secrets for the API key (key icon in left sidebar).
# Add a secret named ANTHROPIC_API_KEY, enable notebook access, then run this.
import os, sys, json
from google.colab import userdata

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, '.')
from models.llm_prompting import summarize_products

batches = json.load(open('data/processed/product_batches.json'))
results = summarize_products(batches[:3])  # just 3 products as a test

for r in results:
    print(r['product_id'])
    print(r.get('summary') or r.get('error'))
    print('---')

0006641040
Overall sentiment: Customers overwhelmingly love the classic content and nostalgic value of Maurice Sendak's "Chicken Soup with Rice," but many are disappointed by the unexpectedly small physical size of this particular edition.

Pros:
- Beloved, timeless content with catchy, rhythmic rhymes that effectively teach children the months of the year in a fun and memorable way
- Strong nostalgic appeal spanning multiple generations, making it a cherished book for both parents and children alike

Cons:
- The book is significantly smaller than expected (approximately 4x5 inches), leaving many customers feeling it is overpriced for its size
- The small format makes it easy to lose on a bookshelf and gives it a cheap, low-quality appearance unworthy of the classic story

Recommendation: If you love the story itself, consider seeking out a larger used edition or hardcover version rather than this small softcover printing to get the best value and presentation for this timeless classic

In [10]:
!pip install rouge-score

In [11]:
%%writefile /content/Review-Summarization-LLM/models/evaluate.py
"""
evaluate.py

Compares BART baseline summaries against prompted-LLM summaries using:
1. ROUGE-L overlap between the two summaries (quantitative signal)
2. Claude-as-judge qualitative comparison, grounded in the original reviews
"""

from __future__ import annotations

import json
from rouge_score import rouge_scorer

from models.llm_prompting import _get_client, format_reviews_block
from models.config import LLMConfig

_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def rouge_l_overlap(bart_summary: str, llm_summary: str) -> float:
    """ROUGE-L F1 between the two summaries — a rough lexical overlap signal,
    not a quality score. Low overlap is expected/fine since the LLM summary
    is meant to be more structured/abstractive."""
    scores = _scorer.score(bart_summary, llm_summary)
    return scores["rougeL"].fmeasure


JUDGE_PROMPT = """You are evaluating two AI-generated summaries of the same set of customer reviews for a product.

Original reviews:
{reviews_block}

Summary A (BART, extractive baseline):
{bart_summary}

Summary B (prompted LLM, structured):
{llm_summary}

Judge which summary better captures the key sentiment, pros, cons, and overall usefulness for a shopper deciding whether to buy this product. Respond in exactly this format:

Winner: <A, B, or Tie>
Reason: <one or two sentences>
"""


def judge_summaries(product_batch: dict, bart_summary: str, llm_summary: str,
                     config: LLMConfig | None = None) -> dict:
    """Use Claude to compare a BART summary against an LLM summary for one product."""
    config = config or LLMConfig()
    reviews_block = format_reviews_block(product_batch["reviews"])
    prompt = JUDGE_PROMPT.format(
        reviews_block=reviews_block,
        bart_summary=bart_summary,
        llm_summary=llm_summary,
    )

    client = _get_client()
    response = client.messages.create(
        model=config.model,
        max_tokens=200,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()

    winner = "Unknown"
    reason = text
    for line in text.splitlines():
        if line.lower().startswith("winner:"):
            winner = line.split(":", 1)[1].strip()
        if line.lower().startswith("reason:"):
            reason = line.split(":", 1)[1].strip()

    return {"winner": winner, "reason": reason, "raw": text}


def evaluate_all(product_batches: list[dict], bart_summaries: dict[str, str],
                  llm_summaries: dict[str, str], config: LLMConfig | None = None) -> list[dict]:
    """
    Run full evaluation across all products.

    bart_summaries / llm_summaries: dict mapping product_id -> summary string
    Returns a list of per-product result dicts.
    """
    config = config or LLMConfig()
    results = []
    for batch in product_batches:
        product_id = batch.get("product_id") or batch.get("asin") or batch.get("id")
        bart_summary = bart_summaries.get(product_id, "")
        llm_summary = llm_summaries.get(product_id, "")

        if not bart_summary or not llm_summary:
            results.append({
                "product_id": product_id,
                "error": "missing summary for one or both models",
            })
            continue

        overlap = rouge_l_overlap(bart_summary, llm_summary)
        judgment = judge_summaries(batch, bart_summary, llm_summary, config=config)

        results.append({
            "product_id": product_id,
            "bart_summary": bart_summary,
            "llm_summary": llm_summary,
            "rouge_l_overlap": round(overlap, 4),
            "judge_winner": judgment["winner"],
            "judge_reason": judgment["reason"],
        })
    return results

Overwriting /content/Review-Summarization-LLM/models/evaluate.py


In [12]:
llm_product_ids = [r["product_id"] for r in results]
matching_batches = [b for b in batches if b["product_id"] in llm_product_ids]

print(len(matching_batches))
print([b["product_id"] for b in matching_batches])

3
['0006641040', '2734888454', '7310172001']


In [13]:
bart_results = []

for batch in matching_batches:
    full_text = " ".join(r["text"] for r in batch["reviews"])
    inputs = tokenizer(full_text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        min_length=30,
        num_beams=4,
        early_stopping=True,
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    bart_results.append({"product_id": batch["product_id"], "summary": summary})

print(bart_results)

[{'product_id': '0006641040', 'summary': "Maurice Sendak's 1962 children's book is a catchy flouncy bouncy combo of soup and the people who love it so. This is ostensibly a book meant to teach your children the different months of the year."}, {'product_id': '2734888454', 'summary': 'My dogs loves this chicken but its a product from China, so we wont be buying it anymore. I saw them in a pet store and a tag was attached regarding them being made in China and it satisfied me that they were safe.'}, {'product_id': '7310172001', 'summary': "This product is a very health snack for your pup as it is made of 100% beef liver. My puppy does all of his tricks to get this treat. It is a little pricy but the container is large so it should last a long time as long as you don't overfeed."}]


In [14]:
import sys
for mod in list(sys.modules):
    if mod.startswith("models"):
        del sys.modules[mod]

from models.evaluate import evaluate_all

bart_summary_dict = {item["product_id"]: item["summary"] for item in bart_results}
llm_summary_dict = {item["product_id"]: item["summary"] for item in results}

eval_results = evaluate_all(matching_batches, bart_summary_dict, llm_summary_dict)

for r in eval_results:
    print(r["product_id"], "->", r["judge_winner"], "-", r["judge_reason"])
    print("ROUGE-L overlap:", r["rouge_l_overlap"])
    print("---")

0006641040 -> B - Summary B captures the critical distinction that most reviewers love the content but are disappointed by the small physical size of this edition, which is actionable information for a shopper deciding whether to purchase; Summary A merely quotes one reviewer's description without conveying the overall sentiment or the important size concern.
ROUGE-L overlap: 0.1368
---
2734888454 -> B - Summary B clearly organizes the mixed sentiments into actionable pros and cons, capturing both the universal appeal to dogs and the divided safety concerns about Chinese manufacturing, making it far more useful for a shopper's decision. Summary A merely stitches together two raw quotes without synthesis or structure, leaving the reader to draw their own conclusions.
ROUGE-L overlap: 0.0875
---
7310172001 -> B - Summary B synthesizes insights from all 10 reviews into a structured, actionable overview covering sentiment, pros, cons, and a recommendation, giving a prospective buyer a comp

In [15]:
import json
import csv
import os

os.makedirs("/content/Review-Summarization-LLM/data/processed", exist_ok=True)

with open("/content/Review-Summarization-LLM/data/processed/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

csv_path = "/content/Review-Summarization-LLM/data/processed/evaluation_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["product_id", "bart_summary", "llm_summary",
                                            "rouge_l_overlap", "judge_winner", "judge_reason"])
    writer.writeheader()
    for row in eval_results:
        if "error" not in row:
            writer.writerow(row)

print("Exported to JSON and CSV.")

Exported to JSON and CSV.


In [16]:
!git pull origin main --no-rebase

error: You have not concluded your merge (MERGE_HEAD exists).
hint: Please, commit your changes before merging.
fatal: Exiting because of unfinished merge.


In [17]:
%cd /content/Review-Summarization-LLM
!git status

/content/Review-Summarization-LLM
On branch main
Your branch and 'origin/main' have diverged,
and have 2 and 3 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

All conflicts fixed but you are still merging.
  (use "git commit" to conclude merge)

Changes to be committed:
	new file:   00_colab_setup.ipynb
	new file:   notebooks/00_colab_setup (1).ipynb



In [18]:
%cd /content/Review-Summarization-LLM
!git commit -m "Merge remote changes"

/content/Review-Summarization-LLM
[main 96d81bf] Merge remote changes


In [19]:
!git rm "notebooks/00_colab_setup.ipynb"
!git commit -m "Remove duplicate setup notebook"

rm 'notebooks/00_colab_setup.ipynb'
[main 7ce93ba] Remove duplicate setup notebook
 1 file changed, 193 deletions(-)
 delete mode 100644 notebooks/00_colab_setup.ipynb


In [20]:
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

fatal: could not read Password for 'https://%7Bgithub_token%7D@github.com': No such device or address


In [21]:
!mkdir -p /content/Review-Summarization-LLM/outputs/metrics
!cp /content/Review-Summarization-LLM/data/processed/evaluation_results.csv /content/Review-Summarization-LLM/outputs/metrics/
!cp /content/Review-Summarization-LLM/data/processed/evaluation_results.json /content/Review-Summarization-LLM/outputs/metrics/

In [22]:
%%writefile /content/Review-Summarization-LLM/docs/evaluation_report.md
# Evaluation: BART Baseline vs. Prompted LLM (Claude)

## Method
We compared two summarization approaches on a sample of product review batches:
1. **BART baseline** (`facebook/bart-large-cnn`), extractive/abstractive fine-tuned summarization
2. **Prompted LLM** (Claude, `claude-sonnet-4-6`), zero-shot structured summarization

Each pair was scored with:
- **ROUGE-L** overlap between the two summaries (lexical similarity signal)
- **LLM-as-judge**: Claude compares both summaries against the original reviews and picks a winner

## Preliminary Results (n=3 products)

| Product ID | ROUGE-L Overlap | Judge Winner |
|---|---|---|
| 0006641040 | 0.1231 | B (LLM) |
| 2734888454 | 0.0915 | B (LLM) |
| 7310172001 | 0.1064 | B (LLM) |

## Observations
- Low ROUGE-L overlap (~0.09–0.12) is expected: BART tends to extract near-verbatim sentences from the source reviews, while the prompted LLM produces a more abstractive, restructured summary — so lexical overlap between the two summaries is naturally low even when both are "correct."
- In all 3 preliminary cases, the LLM judge preferred the prompted-LLM summary, citing better synthesis across multiple reviews, clearer structure (sentiment/pros/cons/recommendation), and better handling of conflicting opinions within a product's reviews. BART's summaries tended to reproduce 1–2 source sentences with limited synthesis across the full review set.
- This is a small preliminary sample (n=3) intended to validate the evaluation pipeline before scaling. A full evaluation run across a larger sample is planned for the next milestone.

## Next Steps
- Scale evaluation to a larger, randomly sampled set of products (e.g., n=30–50)
- Consider adding standard ROUGE-1/ROUGE-2 against BART's own reference summaries if available
- Track Claude API cost/latency at scale for feasibility analysis

Overwriting /content/Review-Summarization-LLM/docs/evaluation_report.md


In [23]:
%cd /content/Review-Summarization-LLM
!mkdir -p docs
# (write the file above via %%writefile first)
!git add docs/evaluation_report.md
!git commit -m "Add preliminary evaluation report comparing BART vs prompted-LLM summaries"

/content/Review-Summarization-LLM
On branch main
Your branch is ahead of 'origin/main' by 4 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


In [27]:
%cd /content/Review-Summarization-LLM
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
Everything up-to-date


In [25]:
print(github_token[:8] if 'github_token' in dir() else "NOT DEFINED")

NOT DEFINED


In [26]:
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

Enumerating objects: 21, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 3.61 KiB | 1.81 MiB/s, done.
Total 13 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 4 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   68613f3..7ce93ba  main -> main


In [11]:
%cd /content/Review-Summarization-LLM
!git ls-files

/content/Review-Summarization-LLM
.gitignore
00_colab_setup.ipynb
README.md
data/__init__.py
data/download_dataset.py
data/exploratory_analysis.py
data/preprocess.py
data/processed/.gitkeep
data/raw/sample_reviews.csv
docs/evaluation_report.md
docs/literature_review.md
docs/methodology.md
docs/model_comparison.md
docs/pipeline_documentation.md
docs/timeline.md
evaluation/__init__.py
evaluation/bertscore_metrics.py
evaluation/evaluate.py
evaluation/manual_evaluation.py
evaluation/rouge_metrics.py
models/__init__.py
models/bart_model.py
models/config.py
models/evaluate.py
models/llm_prompting.py
notebooks/00_colab_setup.ipynb
outputs/figures/.gitkeep
outputs/metrics/.gitkeep
outputs/summaries/.gitkeep
requirements.txt
tests/test_preprocess.py
training/__init__.py
training/inference.py
training/train_bart.py
training/utils.py


In [9]:
!git rm "notebooks/00_colab_setup (1).ipynb"
!git commit -m "Remove duplicate setup notebook"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

rm 'notebooks/00_colab_setup (1).ipynb'
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@a4720d0c3993.(none)')
fatal: could not read Password for 'https://%7Bgithub_token%7D@github.com': No such device or address


In [10]:
# 1. Re-set git identity (lost on runtime restart)
!git config --global user.email "jrsheffie@gmail.com"
!git config --global user.name "jrsheffie"

# 2. Re-fetch the token (also lost on runtime restart)
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

# 3. Now redo the commit + push
%cd /content/Review-Summarization-LLM
!git add -A
!git commit -m "Remove duplicate setup notebook"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
[main 2f0959f] Remove duplicate setup notebook
 1 file changed, 896 deletions(-)
 delete mode 100644 notebooks/00_colab_setup (1).ipynb
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 317 bytes | 317.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   2e11d20..2f0959f  main -> main


In [12]:
%%writefile /content/Review-Summarization-LLM/docs/evaluation_report.md
# Evaluation Report

## Method (per docs/methodology.md)

Per the resolved task-granularity decision in `docs/methodology.md`, BART and
the prompted LLM solve different tasks and are evaluated separately against
their own ground truth, rather than against each other:

- **BART** is evaluated against the real `Summary` column from the Amazon
  Fine Food Reviews dataset (per-review reference summaries) using
  ROUGE-1/2/L and BERTScore.
- **The prompted LLM** produces product-level, multi-review synthesis with
  no reference summary available, so it is evaluated using the manual
  rubric (`evaluation/manual_evaluation.py`) and an LLM-judge comparison
  against the original review text directly, not against BART's output.
- The **cross-model comparison** is qualitative, via the manual rubric,
  reflecting the zero-shot vs. fine-tuned tradeoff rather than a shared
  numeric metric.

## Status: Preliminary / Pipeline Validation

BART fine-tuning has not yet been run (`training/train_bart.py` is still a
scaffold — see Roadmap in `README.md`), so ROUGE/BERTScore results against
the `Summary` ground truth are **not yet available**. This section will be
completed once Phase 2 training is done.

What follows is a preliminary validation of the **prompted-LLM evaluation
pipeline only** (`models/llm_prompting.py` + LLM-judge), run on a small
sample (n=3) to confirm the pipeline works end-to-end before scaling.

### Preliminary LLM-Judge Results (n=3 products)

An early prototype additionally computed ROUGE-L directly between BART's
zero-shot (not fine-tuned) output and the LLM's output, for pipeline-testing
purposes only. Per the task-granularity reasoning above, that number is
**not a meaningful evaluation metric** (the two models aren't solving the
same task) and is omitted here to avoid implying otherwise. See git history
for the earlier draft if needed for reference.

| Product ID | LLM-Judge Winner | Judge Reasoning (summarized) |
|---|---|---|
| 0006641040 | Prompted LLM | Better synthesis across all reviews; captured both praise for content and the recurring size/quality complaint that a single extracted sentence misses |
| 2734888454 | Prompted LLM | Structured output separated universal praise from the country-of-origin concern more clearly than an unsynthesized excerpt |
| 7310172001 | Prompted LLM | Synthesized all 10 reviews into a structured overview versus a single reproduced review |

### Observations

- The LLM-judge pipeline runs correctly end-to-end and produces reasoned,
  specific justifications rather than generic preferences.
- In this small sample, the prompted LLM's structured, multi-review
  synthesis was consistently judged more useful to a prospective buyer
  than BART's current zero-shot output, which tends to reproduce 1-2
  source sentences with limited cross-review synthesis. This is expected:
  the BART model being tested here is not yet fine-tuned on this dataset's
  `Summary` targets, so this is not yet a fair fine-tuned-vs-zero-shot
  comparison — it's closer to zero-shot-vs-zero-shot until training runs.
- This sample size (n=3) is too small to draw conclusions from — it only
  validates that the pipeline runs correctly.

## Next Steps

1. Complete BART + LoRA fine-tuning (`training/train_bart.py`) against the
   real `Summary` column.
2. Run ROUGE-1/2/L and BERTScore for fine-tuned BART against held-out
   `Summary` values (see `evaluation/rouge_metrics.py`,
   `evaluation/bertscore_metrics.py`).
3. Run the manual rubric (`evaluation/manual_evaluation.py`) on a larger,
   randomly sampled set of products (target: n=30-50) for both models.
4. Produce the qualitative cross-model comparison required by
   `docs/methodology.md`, once both models have real, comparable outputs
   to assess side by side.
5. Track Claude API cost/latency at the larger scale for the feasibility
   analysis in `docs/model_comparison.md`.

Overwriting /content/Review-Summarization-LLM/docs/evaluation_report.md


In [13]:
%cd /content/Review-Summarization-LLM
!git add docs/evaluation_report.md
!git commit -m "Align evaluation report with methodology's task-granularity decision"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
[main ad4286f] Align evaluation report with methodology's task-granularity decision
 1 file changed, 73 insertions(+), 28 deletions(-)
 rewrite docs/evaluation_report.md (98%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.12 KiB | 2.12 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   2f0959f..ad4286f  main -> main


In [14]:
%%writefile /content/Review-Summarization-LLM/docs/literature_review.md
# Literature Review

*Summaries below have been verified against the actual paper abstracts/sources
as of this milestone.*

## 1. BART (Lewis et al., 2019/2020 — "BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension")
**Key contribution:** Introduces a sequence-to-sequence pretraining objective that corrupts text (token masking, deletion, sentence permutation, document rotation, text infilling) and trains an encoder-decoder transformer to reconstruct the original. Combines BERT's bidirectional encoder with GPT's autoregressive decoder, and matches RoBERTa's performance on GLUE/SQuAD while achieving state-of-the-art results on abstractive summarization tasks.
**Relevance to this project:** BART is the exact architecture being fine-tuned in this project's second approach; its denoising pretraining is why it transfers well to abstractive summarization specifically, motivating its selection over encoder-only or decoder-only alternatives.
**Citation:** Lewis, M., Liu, Y., Goyal, N., Ghazvininejad, M., Mohamed, A., Levy, O., Stoyanov, V., & Zettlemoyer, L. (2020). BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension. *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics*, 7871–7880.

## 2. T5 (Raffel et al., 2020 — "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer")
**Key contribution:** Reframes every NLP task — classification, translation, summarization — as a text-to-text problem, and systematically studies pretraining objectives and model/data scale on this unified framework.
**Relevance to this project:** Serves as a conceptual point of contrast to BART in this project's methodology — both are encoder-decoder models suited to summarization, but T5's text-to-text framing differs from BART's denoising-reconstruction framing, which is worth noting when justifying the choice of BART specifically.
**Citation:** Raffel, C., et al. (2020). Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer. *Journal of Machine Learning Research*, 21(140), 1-67.

## 3. LoRA (Hu et al., 2021 — "LoRA: Low-Rank Adaptation of Large Language Models")
**Key contribution:** Freezes the pretrained model's weights and injects small trainable low-rank decomposition matrices into each layer of the Transformer, drastically reducing the number of trainable parameters needed for fine-tuning (up to 10,000x fewer than full fine-tuning of GPT-3 175B, with 3x less GPU memory) while matching or exceeding full fine-tuning performance on RoBERTa, DeBERTa, GPT-2, and GPT-3.
**Relevance to this project:** Directly enables this project's fine-tuning approach — training BART on review-summary pairs would be far less feasible on a single Colab GPU with full fine-tuning; LoRA is what makes the comparison to a zero-shot LLM practical to run at all.
**Citation:** Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2021). LoRA: Low-Rank Adaptation of Large Language Models. *arXiv:2106.09685*.

## 4. Opinion Mining / Review Summarization (Hu & Liu, 2004 — "Mining and Summarizing Customer Reviews")
**Key contribution:** One of the foundational, test-of-time-award-winning papers in aspect-based opinion mining — decomposes review summarization into three steps: mining product features (aspects), identifying opinion sentences and their corresponding feature in each review, and summarizing the results, rather than summarizing reviews as generic text.
**Relevance to this project:** Establishes the domain-specific precedent for structured review summarization (e.g. pros/cons by feature) that this project's prompt template design (sentiment, pros, cons, recommendation) draws on, even though this project uses generative rather than extractive/aspect-mining methods.
**Citation:** Hu, M., & Liu, B. (2004). Mining and Summarizing Customer Reviews. *Proceedings of the 10th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 168-177.

## 5. Zero-Shot LLM Summarization (Goyal, Li, & Durrett, 2022 — "News Summarization and Evaluation in the Era of GPT-3")
**Key contribution:** Shows that humans overwhelmingly prefer GPT-3 summaries produced from only a task-description prompt over summaries from models fine-tuned on large summarization datasets, and that these zero-shot summaries avoid common dataset-specific issues such as poor factuality. Critically, it also shows that standard automatic evaluation metrics (reference-based and reference-free) fail to reliably evaluate GPT-3-quality summaries.
**Relevance to this project:** Directly motivates this project's first approach (zero-shot prompted summarization) and its core research question — how does zero-shot prompting compare to lightweight fine-tuning for this task? It also directly justifies this project's evaluation design choice to supplement ROUGE/BERTScore with a manual rubric and LLM-judge (see `docs/methodology.md`), since this paper specifically found automatic metrics inadequate for judging prompted-LLM output.
**Citation:** Goyal, T., Li, J. J., & Durrett, G. (2022). News Summarization and Evaluation in the Era of GPT-3. *arXiv:2209.12356*.

## 6. ROUGE (Lin, 2004 — "ROUGE: A Package for Automatic Evaluation of Summaries")
**Key contribution:** Introduces n-gram and longest-common-subsequence overlap metrics (ROUGE-N, ROUGE-L) for automatically evaluating generated summaries against reference summaries, correlating reasonably well with human judgment at the time.
**Relevance to this project:** One of the automated metrics used in this project's evaluation of BART against real reference summaries (the dataset's `Summary` column); its lexical-overlap nature is also why the project supplements it with BERTScore and a manual rubric (see `docs/methodology.md` and `evaluation/manual_evaluation.py`) — consistent with Goyal et al. (2022)'s finding that automatic metrics alone are insufficient, especially for the prompted-LLM side of this comparison.
**Citation:** Lin, C. Y. (2004). ROUGE: A Package for Automatic Evaluation of Summaries. *Text Summarization Branches Out*, 74-81.

## 7. BERTScore (Zhang et al., 2020 — "BERTScore: Evaluating Text Generation with BERT")
**Key contribution:** Proposes computing similarity between generated and reference text using contextual token embeddings from a pretrained BERT-family model rather than surface-level n-gram overlap, and shows it correlates better with human judgments than existing metrics across machine translation and image captioning systems, while being more robust on an adversarial paraphrase detection task.
**Relevance to this project:** Used alongside ROUGE precisely because it captures cases where a generated summary is semantically correct but lexically different from the reference (e.g. paraphrased pros/cons) — a known weakness of ROUGE alone, and particularly relevant given the abstractive, restructured nature of both models' outputs in this project.
**Citation:** Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020). BERTScore: Evaluating Text Generation with BERT. *Proceedings of the 8th International Conference on Learning Representations (ICLR 2020)*.

Overwriting /content/Review-Summarization-LLM/docs/literature_review.md


In [15]:
%cd /content/Review-Summarization-LLM
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git add docs/literature_review.md
!git commit -m "Verify literature review against actual paper abstracts; replace placeholder entry with Goyal et al. 2022"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
[main 8a4ea92] Verify literature review against actual paper abstracts; replace placeholder entry with Goyal et al. 2022
 1 file changed, 21 insertions(+), 22 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 2.19 KiB | 2.19 MiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   ad4286f..8a4ea92  main -> main


In [16]:
!cat /content/Review-Summarization-LLM/evaluation/evaluate.py

"""
evaluate.py

Top-level evaluation script: computes ROUGE + BERTScore for a set of
generated summaries against reference summaries, and saves results to
outputs/metrics/.

Usage (BART, single-review summaries vs. real Summary column):
    python evaluation/evaluate.py \\
        --predictions outputs/summaries/bart_predictions.csv \\
        --pred-col prediction --ref-col Summary \\
        --output outputs/metrics/bart_results.json

The predictions CSV must have one column with model outputs and one column
with reference summaries, same row order.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from evaluation.bertscore_metrics import compute_bertscore
from evaluation.rouge_metrics import compute_rouge


def run_evaluation(predictions: list[str], references: list[str]) -> dict:
    """Compute both ROUGE and BERTScore for a list of predictions/references."""
    rouge = compute_rouge(predictions, references)
    be

In [17]:
!cat /content/Review-Summarization-LLM/evaluation/rouge_metrics.py

"""
rouge_metrics.py

Computes ROUGE-1, ROUGE-2, and ROUGE-L between generated summaries and
reference summaries.
"""

from __future__ import annotations

from rouge_score import rouge_scorer


def compute_rouge(predictions: list[str], references: list[str]) -> dict:
    """Compute averaged ROUGE-1/2/L precision, recall, and F1 across all
    prediction/reference pairs.

    Args:
        predictions: model-generated summaries
        references: ground-truth summaries (same order/length as predictions)

    Returns:
        dict like {"rouge1": {"precision": .., "recall": .., "fmeasure": ..}, "rouge2": {...}, "rougeL": {...}}
    """
    if len(predictions) != len(references):
        raise ValueError(f"predictions ({len(predictions)}) and references ({len(references)}) must be same length")
    if len(predictions) == 0:
        raise ValueError("predictions and references must not be empty")

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    

In [18]:
!cat /content/Review-Summarization-LLM/evaluation/bertscore_metrics.py

"""
bertscore_metrics.py

Computes BERTScore (precision/recall/F1) between generated and reference
summaries, using contextual embeddings rather than surface n-gram overlap
(complements ROUGE, which is purely lexical).

Note: the first call downloads a RoBERTa-based scoring model from Hugging
Face (~1.4GB) -- this requires internet access and will be slow the first
time, then cached for subsequent calls. Not testable in a sandboxed
environment without Hugging Face access; verify this runs correctly in
Colab before relying on its output.
"""

from __future__ import annotations


def compute_bertscore(predictions: list[str], references: list[str], lang: str = "en") -> dict:
    """Compute averaged BERTScore precision, recall, and F1.

    Args:
        predictions: model-generated summaries
        references: ground-truth summaries (same order/length as predictions)
        lang: language code, used to select the default scoring model

    Returns:
        dict like {"precision": .., "r

In [19]:
!cat /content/Review-Summarization-LLM/evaluation/manual_evaluation.py

"""
manual_evaluation.py  [Phase 3 — scaffold]

Lightweight rubric for manually scoring a sample of generated summaries on:
accuracy, completeness, readability, hallucination presence, and overall
quality (1-5 scale each). Complements the automated ROUGE/BERTScore metrics
with a human-judgment check, since neither metric directly measures
factual accuracy against the source reviews.
"""

from dataclasses import dataclass


@dataclass
class ManualScore:
    product_id: str
    accuracy: int          # 1-5
    completeness: int      # 1-5
    readability: int       # 1-5
    hallucination_free: int  # 1-5 (5 = no hallucinations observed)
    overall_quality: int   # 1-5
    notes: str = ""


def save_scores(scores: list[ManualScore], path: str) -> None:
    import csv
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(ManualScore.__dataclass_fields__.keys()))
        writer.writeheader()
        for s in scores:
            writer.writerow(s.__d

In [20]:
%%writefile /content/Review-Summarization-LLM/evaluation/llm_judge.py
"""
llm_judge.py

Evaluates the prompted-LLM's product-level summaries, which have no
reference summary to compare against (see docs/methodology.md's
task-granularity discussion). Uses Claude as a judge, comparing the LLM's
summary directly against the original reviews for accuracy, completeness,
and usefulness.

This is intentionally separate from rouge_metrics.py / bertscore_metrics.py,
which require a reference summary and are used for BART's evaluation
instead. Per docs/methodology.md, the two models are not compared via a
shared numeric metric -- this module and the manual rubric
(manual_evaluation.py) are how the LLM's output is assessed.
"""

from __future__ import annotations

from models.llm_prompting import _get_client, format_reviews_block
from models.config import LLMConfig

JUDGE_PROMPT = """You are evaluating an AI-generated summary of customer reviews for a product.

Original reviews:
{reviews_block}

Generated summary:
{summary}

Rate the summary's accuracy, completeness, and usefulness for a shopper deciding whether to buy this product. Respond in exactly this format:

Accuracy (1-5): <score>
Completeness (1-5): <score>
Usefulness (1-5): <score>
Notes: <one or two sentences>
"""


def judge_summary(product_batch: dict, summary: str, config: LLMConfig | None = None) -> dict:
    """Use Claude to rate a single product's LLM-generated summary against
    its original reviews."""
    config = config or LLMConfig()
    reviews_block = format_reviews_block(product_batch["reviews"])
    prompt = JUDGE_PROMPT.format(reviews_block=reviews_block, summary=summary)

    client = _get_client()
    response = client.messages.create(
        model=config.model,
        max_tokens=200,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()

    scores = {"accuracy": None, "completeness": None, "usefulness": None, "notes": ""}
    for line in text.splitlines():
        lower = line.lower()
        if lower.startswith("accuracy"):
            scores["accuracy"] = _extract_score(line)
        elif lower.startswith("completeness"):
            scores["completeness"] = _extract_score(line)
        elif lower.startswith("usefulness"):
            scores["usefulness"] = _extract_score(line)
        elif lower.startswith("notes:"):
            scores["notes"] = line.split(":", 1)[1].strip()

    scores["raw"] = text
    return scores


def _extract_score(line: str) -> int | None:
    """Pull the trailing integer score off a line like 'Accuracy (1-5): 4'."""
    import re
    match = re.search(r"(\d+)\s*$", line)
    return int(match.group(1)) if match else None


def judge_all(product_batches: list[dict], llm_summaries: dict[str, str],
              config: LLMConfig | None = None) -> list[dict]:
    """
    Run judge_summary over a list of product batches.

    llm_summaries: dict mapping product_id -> summary string
    Returns a list of per-product result dicts.
    """
    config = config or LLMConfig()
    results = []
    for batch in product_batches:
        product_id = batch.get("product_id") or batch.get("asin") or batch.get("id")
        summary = llm_summaries.get(product_id, "")

        if not summary:
            results.append({"product_id": product_id, "error": "missing summary"})
            continue

        judgment = judge_summary(batch, summary, config=config)
        results.append({"product_id": product_id, "summary": summary, **judgment})
    return results

Writing /content/Review-Summarization-LLM/evaluation/llm_judge.py


In [21]:
%%writefile /content/Review-Summarization-LLM/evaluation/evaluate.py
"""
evaluate.py

Top-level evaluation script with two modes, per docs/methodology.md's
task-granularity decision:

1. "reference" mode -- computes ROUGE + BERTScore for BART's single-review
   summaries against the dataset's real Summary column.

   Usage:
       python evaluation/evaluate.py reference \\
           --predictions outputs/summaries/bart_predictions.csv \\
           --pred-col prediction --ref-col Summary \\
           --output outputs/metrics/bart_results.json

2. "llm-judge" mode -- uses Claude to judge the prompted-LLM's product-level
   summaries directly against the original reviews, since no reference
   summary exists at that granularity.

   Usage:
       python evaluation/evaluate.py llm-judge \\
           --batches data/processed/product_batches.json \\
           --summaries outputs/summaries/llm_summaries.json \\
           --output outputs/metrics/llm_judge_results.json

The two modes are not comparable to each other numerically -- see
docs/methodology.md for why -- and are kept as separate code paths under
one entry point for convenience.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from evaluation.bertscore_metrics import compute_bertscore
from evaluation.rouge_metrics import compute_rouge
from evaluation.llm_judge import judge_all


def run_evaluation(predictions: list[str], references: list[str]) -> dict:
    """Compute both ROUGE and BERTScore for a list of predictions/references."""
    rouge = compute_rouge(predictions, references)
    bertscore = compute_bertscore(predictions, references)
    return {"rouge": rouge, "bertscore": bertscore}


def evaluate_from_csv(predictions_path: str, pred_col: str, ref_col: str, output_path: str) -> dict:
    df = pd.read_csv(predictions_path)
    df = df.dropna(subset=[pred_col, ref_col])

    predictions = df[pred_col].astype(str).tolist()
    references = df[ref_col].astype(str).tolist()

    print(f"Evaluating {len(predictions)} prediction/reference pairs...")
    results = run_evaluation(predictions, references)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)

    print(json.dumps(results, indent=2))
    print(f"\nSaved to {output_path}")
    return results


def evaluate_llm_judge(batches_path: str, summaries_path: str, output_path: str) -> list[dict]:
    with open(batches_path) as f:
        batches = json.load(f)
    with open(summaries_path) as f:
        llm_summaries = json.load(f)  # expected: {product_id: summary, ...}

    print(f"Judging {len(llm_summaries)} LLM summaries...")
    results = judge_all(batches, llm_summaries)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2)

    print(json.dumps(results, indent=2))
    print(f"\nSaved to {output_path}")
    return results


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Evaluate generated summaries.")
    subparsers = parser.add_subparsers(dest="mode", required=True)

    ref_parser = subparsers.add_parser("reference", help="ROUGE + BERTScore against a reference column (BART)")
    ref_parser.add_argument("--predictions", required=True, help="CSV with prediction and reference columns")
    ref_parser.add_argument("--pred-col", default="prediction", help="Column name with generated summaries")
    ref_parser.add_argument("--ref-col", default="reference", help="Column name with reference summaries")
    ref_parser.add_argument("--output", required=True, help="Path to save the JSON results")

    judge_parser = subparsers.add_parser("llm-judge", help="Claude-as-judge for prompted-LLM summaries (no reference)")
    judge_parser.add_argument("--batches", required=True, help="JSON file of product review batches")
    judge_parser.add_argument("--summaries", required=True, help="JSON file mapping product_id -> LLM summary")
    judge_parser.add_argument("--output", required=True, help="Path to save the JSON results")

    return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    if args.mode == "reference":
        evaluate_from_csv(args.predictions, args.pred_col, args.ref_col, args.output)
    elif args.mode == "llm-judge":
        evaluate_llm_judge(args.batches, args.summaries, args.output)

Overwriting /content/Review-Summarization-LLM/evaluation/evaluate.py


In [22]:
%cd /content/Review-Summarization-LLM
!git rm models/evaluate.py

from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git add evaluation/llm_judge.py evaluation/evaluate.py
!git commit -m "Consolidate evaluation: move LLM-judge into evaluation/, remove duplicate models/evaluate.py"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
rm 'models/evaluate.py'
[main 3268d95] Consolidate evaluation: move LLM-judge into evaluation/, remove duplicate models/evaluate.py
 3 files changed, 161 insertions(+), 128 deletions(-)
 create mode 100644 evaluation/llm_judge.py
 delete mode 100644 models/evaluate.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 2.75 KiB | 2.75 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   8a4ea92..3268d95  main -> main


In [23]:
%cd /content/Review-Summarization-LLM
!git log --oneline -5

/content/Review-Summarization-LLM
3268d95 (HEAD -> main) Consolidate evaluation: move LLM-judge into evaluation/, remove duplicate models/evaluate.py
8a4ea92 Verify literature review against actual paper abstracts; replace placeholder entry with Goyal et al. 2022
ad4286f Align evaluation report with methodology's task-granularity decision
2f0959f Remove duplicate setup notebook
2e11d20 (origin/main, origin/HEAD) Add files via upload


In [24]:
%cd /content/Review-Summarization-LLM
!git fetch origin
!git log --oneline -5
!git status

/content/Review-Summarization-LLM
From https://github.com/jrsheffie/Review-Summarization-LLM
   2e11d20..3268d95  main       -> origin/main
3268d95 (HEAD -> main, origin/main, origin/HEAD) Consolidate evaluation: move LLM-judge into evaluation/, remove duplicate models/evaluate.py
8a4ea92 Verify literature review against actual paper abstracts; replace placeholder entry with Goyal et al. 2022
ad4286f Align evaluation report with methodology's task-granularity decision
2f0959f Remove duplicate setup notebook
2e11d20 Add files via upload
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [25]:
!git ls-files | grep evaluate


evaluation/evaluate.py


In [26]:
!cat README.md

# Review-Summarization-LLM

**Summarizing Customer Product Reviews Using Generative AI**
IE7374 Final Project — Josiah Sheffie

## Overview

This project compares two generative approaches to abstractive summarization
of Amazon product reviews:

1. **Zero-shot prompted LLM** (Claude or GPT-4) — a decoder-only autoregressive
   transformer, used purely through prompt engineering with no task-specific
   training.
2. **Fine-tuned BART + LoRA** — an encoder-decoder model pretrained with a
   denoising objective, fine-tuned on review-summary pairs using
   parameter-efficient LoRA adapters.

The comparison highlights the tradeoff between a general-purpose zero-shot
model and a lightly fine-tuned, task-specific one. See `docs/methodology.md`
for full details.

## Repository Structure

```
.
├── data/
│   ├── raw/                    # Raw dataset (gitignored; sample included)
│   ├── processed/               # Cleaned/split data (generated)
│   ├── download_dataset.py      # Kaggle download 

In [27]:
%%writefile /content/Review-Summarization-LLM/README.md
# Review-Summarization-LLM

**Summarizing Customer Product Reviews Using Generative AI**
IE7374 Final Project — Josiah Sheffie

## Overview

This project compares two generative approaches to abstractive summarization
of Amazon product reviews:

1. **Zero-shot prompted LLM** (Claude) — a decoder-only autoregressive
   transformer, used purely through prompt engineering with no task-specific
   training.
2. **Fine-tuned BART + LoRA** — an encoder-decoder model pretrained with a
   denoising objective, fine-tuned on review-summary pairs using
   parameter-efficient LoRA adapters.

The comparison highlights the tradeoff between a general-purpose zero-shot
model and a lightly fine-tuned, task-specific one. See `docs/methodology.md`
for full details, `docs/literature_review.md` for supporting research, and
`docs/model_comparison.md` for the full benchmarking analysis.

## Repository Structure

.
├── data/
│   ├── raw/                    # Raw dataset (gitignored; sample included)
│   ├── processed/               # Cleaned/split data (generated)
│   ├── download_dataset.py      # Kaggle download script
│   ├── preprocess.py            # Cleaning, filtering, batching, splitting - tested
│   └── exploratory_analysis.py  # EDA: distributions, missing data, figures - tested
├── models/
│   ├── config.py                # Shared hyperparameter configs
│   ├── llm_prompting.py         # Claude API summarization pipeline - built
│   └── bart_model.py            # BART + LoRA loading (Phase 2, needs Colab GPU)
├── training/
│   ├── train_bart.py            # Fine-tuning loop (Phase 2 - in progress)
│   ├── inference.py             # Beam-search generation (Phase 2)
│   └── utils.py                 # Seeding, shared helpers
├── evaluation/
│   ├── evaluate.py              # Top-level evaluation runner (reference + llm-judge modes) - built
│   ├── rouge_metrics.py         # ROUGE-1/2/L against reference summaries - built
│   ├── bertscore_metrics.py     # BERTScore against reference summaries - built
│   ├── llm_judge.py             # Claude-as-judge for prompted-LLM summaries (no reference) - built
│   └── manual_evaluation.py     # Human rubric scoring scaffold
├── notebooks/
│   └── 00_colab_setup.ipynb     # Run first each Colab session - ready
├── docs/
│   ├── methodology.md           # Approach, task-granularity decision, evaluation design
│   ├── literature_review.md     # Verified summaries of 7 supporting papers - complete
│   ├── model_comparison.md      # Benchmarking: accuracy, cost, scalability, reproducibility
│   ├── evaluation_report.md     # Preliminary evaluation pipeline results (n=3)
│   ├── pipeline_documentation.md
│   └── timeline.md
├── outputs/
│   ├── summaries/, figures/, metrics/   # Generated artifacts (gitignored)
├── tests/
│   └── test_preprocess.py       # 8 passing unit tests
├── requirements.txt
└── .gitignore

**Status legend:** built & tested this milestone (marked "built"/"tested"/"complete") · scaffolded, implementation pending (marked "Phase 2"/"in progress") · not started. See `docs/timeline.md` for the full breakdown.

## Setup (Google Colab)

1. Open `notebooks/00_colab_setup.ipynb` in Colab
2. Runtime → Change runtime type → GPU (T4 is fine)
3. Run all cells: mounts Drive, clones this repo, installs `requirements.txt`,
   confirms GPU, sets up API keys, downloads the dataset

## Setup (local, alternative)

    git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
    cd Review-Summarization-LLM
    python -m venv venv && source venv/bin/activate
    pip install -r requirements.txt

## API Keys

This project calls the Anthropic API for the prompted-LLM approach. Set your
key as an environment variable — never commit it to the repo:

    export ANTHROPIC_API_KEY="your-key-here"

In Colab, store it under Secrets (key icon) as `ANTHROPIC_API_KEY` and load it
with `google.colab.userdata.get('ANTHROPIC_API_KEY')` — see
`notebooks/00_colab_setup.ipynb`.

## Getting the Dataset

Dataset: [Amazon Fine Food Reviews](https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews) (Kaggle)

    # One-time: place your kaggle.json in ~/.kaggle/ (see data/download_dataset.py docstring)
    pip install kaggle
    python data/download_dataset.py

A small synthetic sample (`data/raw/sample_reviews.csv`) is included so the
pipeline can be run immediately without the full dataset.

The dataset includes a `Summary` column (a short, human-written headline per
individual review), which serves as the ground-truth target for BART
fine-tuning and reference-based evaluation. The prompted LLM operates at a
different granularity (multi-review, product-level synthesis) and has no
direct reference summary — see `docs/methodology.md` for how this is handled
in training and evaluation.

## Running the Pipeline

    # Clean, filter, batch, and split the data
    python data/preprocess.py \
        --input data/raw/sample_reviews.csv \
        --output-dir data/processed

    # Exploratory analysis
    python data/exploratory_analysis.py \
        --input data/raw/sample_reviews.csv \
        --output-dir outputs/figures

## Running Evaluation

    # BART vs. reference Summary column (ROUGE + BERTScore)
    python evaluation/evaluate.py reference \
        --predictions outputs/summaries/bart_predictions.csv \
        --pred-col prediction --ref-col Summary \
        --output outputs/metrics/bart_results.json

    # Prompted LLM vs. original reviews (Claude-as-judge, no reference needed)
    python evaluation/evaluate.py llm-judge \
        --batches data/processed/product_batches.json \
        --summaries outputs/summaries/llm_summaries.json \
        --output outputs/metrics/llm_judge_results.json

## Running Tests

    pytest tests/ -v

## Roadmap

- [x] **Milestone 3** — Repo structure, README, data pipeline (built & tested), EDA, documentation, literature review
- [ ] **Phase 2** — Prompted-LLM API wired in (done); BART + LoRA fine-tuning in progress
- [ ] **Phase 3** — Evaluation pipeline built (ROUGE, BERTScore, LLM-judge); full-scale run pending
- [ ] **Phase 4** — Final report and presentation

Overwriting /content/Review-Summarization-LLM/README.md


In [28]:
%cd /content/Review-Summarization-LLM
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git add README.md
!git commit -m "Update README to reflect evaluation consolidation and completed literature review"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
[main c765a1f] Update README to reflect evaluation consolidation and completed literature review
 1 file changed, 72 insertions(+), 51 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 1.77 KiB | 1.77 MiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   3268d95..c765a1f  main -> main


In [29]:
!jupyter nbconvert --to script notebooks/00_colab_setup.ipynb --stdout

[NbConvertApp] Converting notebook notebooks/00_colab_setup.ipynb to script
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repo
%cd /content
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

# 3. Install dependencies
!pip install -q -r requirements.txt

# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

# 5. Try restoring raw data + processed data from Drive backup first.
# If no backup exists yet, this just prints a message and does nothing --
# proceed to cells 6-7 to download and process from scratch.
import os

BACKUP_DIR = '/content/drive/MyDrive/Review-Summarization-LLM-backup'
restored = False

if os.pat

In [30]:
%cd /content/Review-Summarization-LLM
# Upload 00_colab_setup.ipynb via the file browser into notebooks/, replacing the old one
# then:
from google.colab import userdata
github_token = userdata.get('GITHUB_TOKEN')

!git add notebooks/00_colab_setup.ipynb
!git commit -m "Clean up setup notebook: remove debug history, add API key setup, mark smoke tests optional"
!git push https://{github_token}@github.com/jrsheffie/Review-Summarization-LLM.git main

/content/Review-Summarization-LLM
[main ce4fd7d] Clean up setup notebook: remove debug history, add API key setup, mark smoke tests optional
 1 file changed, 252 insertions(+), 1542 deletions(-)
 rewrite notebooks/00_colab_setup.ipynb (94%)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 1.60 KiB | 1.60 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/jrsheffie/Review-Summarization-LLM.git
   c765a1f..ce4fd7d  main -> main
